## 1. Imports

Only sklearn / stdlib / `paths`. Everything domain-specific is defined inline below.

In [ ]:
def build_preprocessor() -> ColumnTransformer:
    """Four parallel branches, all refit per CV fold."""
    ohe_branch = Pipeline([
        # Fill NaN with sentinel BEFORE encoding — makes missingness an explicit OHE level.
        ("fill_missing", SimpleImputer(strategy="constant", fill_value=MISSING_CATEGORY_SENTINEL)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=int)),
    ])

    # PriorEncoder gates the rolling cancellation rate on `label_resolution_date`
    # (FWD-issue date, or termination date for non-FWD outcomes) rather than on
    # petition_filing_date. A training trial whose petition pre-dates T₀ but whose
    # FWD post-dates T₀ has a label that wasn't observable at T₀, so including it
    # in the rolling rate would leak future information. See `prediction_scope.md` §4.
    prior_branches = [
        (
            f"prior_{'_'.join(group)}",
            PriorEncoder(
                group_columns=list(group),
                date_column=PRIOR_DATE_COLUMN,
                resolution_date_column=PRIOR_RESOLUTION_DATE_COLUMN,
            ),
            [*group, PRIOR_DATE_COLUMN, PRIOR_RESOLUTION_DATE_COLUMN],
        )
        for group in PRIOR_GROUP_COLUMNS
    ]

    return ColumnTransformer(
        transformers=[
            *prior_branches,
            ("freq", FrequencyEncoder(), list(FREQUENCY_CATEGORICAL_COLUMNS)),
            ("ohe", ohe_branch, list(OHE_CATEGORICAL_COLUMNS)),
            ("num_impute", SimpleImputer(strategy="median"), make_column_selector(dtype_include=np.number)),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_pipeline(estimator: BaseEstimator) -> Pipeline:
    """Compose the preprocessor with `estimator` into a refit-per-fold pipeline."""
    return Pipeline([("preprocess", build_preprocessor()), ("model", estimator)])

## 4. Training primitives — split, CV, evaluation

In [ ]:
def time_split(X, y, petition_dates, *, holdout_after):
    """Split (X, y, dates) into train (< cutoff) and held-out test (≥ cutoff).

    The held-out tail is the deployment-honest evaluation slice — never
    inspected during selection or hyperparameter search.
    """
    cutoff = pd.Timestamp(holdout_after)
    dates = pd.to_datetime(petition_dates)
    train_mask = dates < cutoff
    test_mask = ~train_mask
    return (
        X.loc[train_mask], X.loc[test_mask],
        y.loc[train_mask], y.loc[test_mask],
        dates.loc[train_mask], dates.loc[test_mask],
    )


DEFAULT_TIME_SERIES_SCORING = ("roc_auc", "average_precision", "f1")


def _date_safe_folds(sorted_dates, n_splits):
    """Forward-walking train/test indices that snap to date transitions.

    Same chunk layout as sklearn's TimeSeriesSplit but pushes each cut
    forward to the first row whose date is strictly greater than the
    previous row's. Same-day rows therefore never straddle a fold
    boundary — joinder cases and multi-petition campaigns are filed in
    same-day batches.
    """
    n = len(sorted_dates)
    chunk = n // (n_splits + 1)

    def snap(idx):
        if idx <= 0:
            return 0
        if idx >= n:
            return n
        return int(np.searchsorted(sorted_dates, sorted_dates[idx - 1], side="right"))

    folds = []
    for k in range(1, n_splits + 1):
        train_end = snap(k * chunk)
        test_end = snap((k + 1) * chunk) if k < n_splits else n
        if train_end >= test_end:
            continue
        folds.append((np.arange(0, train_end), np.arange(train_end, test_end)))
    return folds


def time_series_cv(pipeline, X, y, petition_dates, n_splits=5, scoring=DEFAULT_TIME_SERIES_SCORING):
    """Forward-walking time-series CV against petition_filing_date."""
    order = np.argsort(pd.to_datetime(petition_dates).to_numpy(), kind="stable")
    X_sorted = X.iloc[order].reset_index(drop=True)
    y_sorted = y.iloc[order].reset_index(drop=True)
    sorted_dates = pd.to_datetime(petition_dates).to_numpy()[order]
    return cross_validate(
        pipeline, X_sorted, y_sorted,
        cv=_date_safe_folds(sorted_dates, n_splits),
        scoring=list(scoring),
        n_jobs=-1,
    )

In [ ]:
@dataclass
class ModelMetrics:
    model_name: str
    accuracy: float
    roc_auc: float
    classification_report: dict


def evaluate_model(model, X_test, y_test, model_name):
    """Compute held-out metrics. CV picks the model; this number is what you report."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return ModelMetrics(
        model_name=model_name,
        accuracy=accuracy_score(y_test, y_pred),
        roc_auc=roc_auc_score(y_test, y_prob),
        classification_report=classification_report(y_test, y_pred, output_dict=True),
    )

## 5. Notebook helpers — loaders, orchestrator, plot fns

In [ ]:
def _resolve(processed_dir):
    proc = Path(processed_dir) if processed_dir else paths.processed_dir()
    if not proc.exists():
        raise FileNotFoundError(
            f"{proc} not found. Generate parquets via `drivers/run_features.py` after the upstream stages, or copy them in."
        )
    return proc


def load_features(processed_dir=None):
    return pd.read_parquet(_resolve(processed_dir) / "features.parquet")


def load_joined(processed_dir=None):
    return pd.read_parquet(_resolve(processed_dir) / "joined_trials.parquet")


def merge_features_with_labels(features, joined, *, mature_days=540, today=None):
    """Inner-join features ⨝ (label + T₀ + patent_number + label_resolution_date) + maturity filter.

    `label_resolution_date` is `decision_issue_date` for FWD-resolved trials,
    else `termination_date`. PriorEncoder uses it (not petition_filing_date) to
    order its rolling cumsum so a training row only contributes once its label
    actually crystallized — see `prediction_scope.md` §4.

    Drops trials whose petition is more recent than `today - mature_days`
    — their `cancelled` label hasn't crystallized yet (~18 mo for FWDs).
    """
    label_cols = joined[
        [
            "trial_number",
            "cancelled",
            "petition_filing_date",
            "patent_number",
            "decision_issue_date",
            "termination_date",
        ]
    ].copy()
    label_cols["trial_number"] = label_cols["trial_number"].astype(str)
    label_cols["patent_number"] = label_cols["patent_number"].astype(str)
    label_cols["label_resolution_date"] = pd.to_datetime(
        label_cols["decision_issue_date"], errors="coerce"
    ).combine_first(
        pd.to_datetime(label_cols["termination_date"], errors="coerce")
    )
    label_cols = label_cols.drop(columns=["decision_issue_date", "termination_date"])
    features = features.copy()
    features["trial_number"] = features["trial_number"].astype(str)
    merged = features.merge(label_cols, on="trial_number", how="inner")
    if mature_days > 0:
        now = pd.Timestamp(today) if today else pd.Timestamp(pd.Timestamp.today().date())
        cutoff = now - pd.Timedelta(days=mature_days)
        merged = merged[pd.to_datetime(merged["petition_filing_date"]) <= cutoff].copy()
    return merged

In [ ]:
@dataclass
class TrainingConfig:
    holdout_after: str | pd.Timestamp
    model_name: ModelName = ModelName.RANDOM_FOREST
    mature_days: int = 540
    cv_folds: int = 5
    today: str | pd.Timestamp | None = None


@dataclass
class TrainingResult:
    pipeline: Pipeline
    cv_results: dict[str, Any]
    metrics: ModelMetrics
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    dates_train: pd.Series
    dates_test: pd.Series


def run_training(config, *, features=None, joined=None, estimator_factory=None):
    """Mirror of drivers/run_train.py. Pass `estimator_factory` to swap the estimator without mutating MODELS."""
    if features is None:
        features = load_features()
    if joined is None:
        joined = load_joined()
    merged = merge_features_with_labels(
        features, joined, mature_days=config.mature_days, today=config.today
    )
    y = merged["cancelled"].astype(int)
    dates = merged["petition_filing_date"]
    # petition_filing_date + patent_number + label_resolution_date stay in X —
    # PriorEncoder reads all three (T₀ for the query, group key, resolution date
    # for the leakage-safe cumsum); other branches' selectors ignore them and
    # remainder="drop" strips them downstream.
    X = merged.drop(columns=["trial_number", "cancelled"])
    # Held-out tail is the deployment-honest slice — never seen by CV or selection.
    X_train, X_test, y_train, y_test, dates_train, dates_test = time_split(
        X, y, dates, holdout_after=config.holdout_after
    )

    # Preprocessor refits inside time_series_cv per fold — no leakage from test rows.
    factory = estimator_factory or MODELS[config.model_name]
    pipeline = build_pipeline(factory())
    cv = time_series_cv(pipeline, X_train, y_train, dates_train, n_splits=config.cv_folds)
    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(pipeline, X_test, y_test, config.model_name.value)

    return TrainingResult(
        pipeline=pipeline, cv_results=cv, metrics=metrics,
        X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test,
        dates_train=dates_train, dates_test=dates_test,
    )

In [ ]:
# Plot helpers — return matplotlib figures for inline display.

def plot_roc(models, X_test, y_test):
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve

    fig, ax = plt.subplots(figsize=(8, 6))
    for name, model in models.items():
        prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set(xlabel="FPR", ylabel="TPR", title="ROC")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_confusion(model, X_test, y_test, *, name="model"):
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y_test, model.predict(X_test))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    ax.set(
        xticks=[0, 1], yticks=[0, 1],
        xticklabels=["not cancelled", "cancelled"],
        yticklabels=["not cancelled", "cancelled"],
        xlabel="predicted", ylabel="actual", title=f"Confusion — {name}",
    )
    fig.tight_layout()
    return fig


def plot_feature_importance(pipeline, *, top_n=20, name="model"):
    import matplotlib.pyplot as plt

    pre = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]
    feat_names = list(pre.get_feature_names_out())
    if hasattr(model, "feature_importances_"):
        importances = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        importances = np.abs(np.asarray(model.coef_).ravel())
    else:
        raise AttributeError(f"{type(model).__name__} has no feature_importances_ or coef_")
    idx = np.argsort(importances)[-top_n:]
    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.25)))
    ax.barh(range(len(idx)), importances[idx])
    ax.set_yticks(range(len(idx)))
    ax.set_yticklabels([feat_names[i] for i in idx])
    ax.set(xlabel="importance", title=f"Top {top_n} features — {name}")
    fig.tight_layout()
    return fig

## 6. Load the data

`features` is the model-ready matrix from `drivers/run_features.py`; `joined` is the master frame (raw cols + label + T₀).

In [ ]:
features = load_features()
joined = load_joined()
print(f"features: {features.shape}, joined: {joined.shape}")
features.head()

## 7. Quickstart — one-liner end-to-end

`TrainingConfig` carries every knob; `run_training` does merge → maturity-filter → time-split → CV → refit → held-out evaluation.

In [ ]:
config = TrainingConfig(
    holdout_after="2024-06-01",
    model_name=ModelName.RANDOM_FOREST,
    mature_days=540,
    cv_folds=5,
)
result = run_training(config, features=features, joined=joined)

cv_auc = result.cv_results["test_roc_auc"]
print(f"train rows: {len(result.X_train)}, held-out: {len(result.X_test)}")
print(f"CV ROC-AUC:    {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")
print(f"held-out AUC:  {result.metrics.roc_auc:.4f}")
print(f"held-out acc:  {result.metrics.accuracy:.4f}")

## 8. Step-by-step — same flow, peeled into editable cells

In [ ]:
# 8a. Merge + maturity filter
merged = merge_features_with_labels(features, joined, mature_days=540)
print(f"after maturity filter: {len(merged)} rows")

In [ ]:
# 8b. Time split — held-out tail vs trainable history.
y = merged["cancelled"].astype(int)
dates = merged["petition_filing_date"]
# Keep petition_filing_date + patent_number + label_resolution_date in X — PriorEncoder
# reads all three; other branches' selectors ignore them.
X = merged.drop(columns=["trial_number", "cancelled"])

# Date-based split. Random K-fold would let later cancellations train models that score earlier rows.
X_train, X_test, y_train, y_test, dates_train, dates_test = time_split(
    X, y, dates, holdout_after="2024-06-01"
)
print(f"train: {len(X_train)}, test: {len(X_test)}")

In [ ]:
# 8c. Forward-walking CV — each test fold sits strictly after its train rows by date.
pipeline = build_pipeline(MODELS[ModelName.RANDOM_FOREST]())
cv = time_series_cv(pipeline, X_train, y_train, dates_train, n_splits=5)
print(f"CV ROC-AUC: {cv['test_roc_auc'].mean():.4f} +/- {cv['test_roc_auc'].std():.4f}")

In [ ]:
# 8d. Refit on full train, evaluate once on held-out.
# CV is for model selection; this number is what you report.
pipeline.fit(X_train, y_train)
metrics = evaluate_model(pipeline, X_test, y_test, "random_forest")
metrics

In [ ]:
plot_roc({"random_forest": pipeline}, X_test, y_test);

In [ ]:
plot_confusion(pipeline, X_test, y_test, name="random_forest");

In [ ]:
plot_feature_importance(pipeline, top_n=20, name="random_forest");

## 9. Overriding the estimator

Two non-mutating patterns: pass an `estimator_factory` callable to `run_training`, or build a local `MODELS` dict that adds entries on top of the §2 one. The inline `MODELS` stays untouched.

In [ ]:
# 9a. Inline factory: tuned RF
def tuned_rf():
    return RandomForestClassifier(
        n_estimators=500, max_depth=15, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )

tuned = run_training(
    TrainingConfig(holdout_after="2024-06-01", model_name=ModelName.RANDOM_FOREST),
    features=features, joined=joined,
    estimator_factory=tuned_rf,
)
print(f"tuned RF held-out AUC: {tuned.metrics.roc_auc:.4f}")

In [ ]:
# 9b. Local MODELS dict — adds an entry without mutating the §2 dict.
from sklearn.linear_model import LogisticRegression

MY_MODELS = {
    **MODELS,
    "logistic_regression": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
}
lr_pipe = build_pipeline(MY_MODELS["logistic_regression"]())
lr_cv = time_series_cv(lr_pipe, X_train, y_train, dates_train, n_splits=5)
print(f"LR CV ROC-AUC: {lr_cv['test_roc_auc'].mean():.4f} +/- {lr_cv['test_roc_auc'].std():.4f}")

## 10. Overriding the preprocessor

`build_preprocessor` is the default. To customize, build your own `ColumnTransformer` and wrap it in a `Pipeline` — `time_series_cv` refits the whole composite per fold either way.

In [ ]:
# Drop priors, keep frequency + OHE + median impute.
MY_OHE = ("technology_center", "cpc_section")
MY_FREQ = ("petitioner_real_party", "owner_real_party")

ohe_branch = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value=MISSING_CATEGORY_SENTINEL)),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=int)),
])

my_preprocessor = ColumnTransformer(
    transformers=[
        ("freq", FrequencyEncoder(), list(MY_FREQ)),
        ("ohe", ohe_branch, list(MY_OHE)),
        ("num", SimpleImputer(strategy="median"), make_column_selector(dtype_include=np.number)),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Pipeline wrapper is what makes the custom preprocessor refit per CV fold.
no_prior_pipe = Pipeline([("preprocess", my_preprocessor), ("model", MODELS[ModelName.RANDOM_FOREST]())])
no_prior_cv = time_series_cv(no_prior_pipe, X_train, y_train, dates_train, n_splits=5)
print(f"no-prior RF CV ROC-AUC: {no_prior_cv['test_roc_auc'].mean():.4f} (vs default w/ priors)")

## 11. Comparing estimators

Side-by-side CV on the same train slice. Pick the highest stable mean, then run §8d once on the held-out tail.

In [ ]:
rows = []
for name in (ModelName.RANDOM_FOREST,):
    pipe = build_pipeline(MODELS[name]())
    cv_i = time_series_cv(pipe, X_train, y_train, dates_train, n_splits=5)
    rows.append({
        "model": name.value,
        "cv_roc_auc_mean": cv_i["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_i["test_roc_auc"].std(),
    })
pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)